# 01. Data Cleaning
In questo notebook viene implementata la pipeline iniziale per il caricamento e la pulizia dei dati relativi al mondo degli anime.   
I dati 'sanificati' vengono poi salvati nella cartella `data/processed/` per le successive fasi di analisi e visualizzazione.

### Obbiettivi
- **Caricamento dei CSV**: importazione dei file necessari per l'analisi.
- **Rimozione dei Duplicati**: pulizia dei valori duplicati.
- **Gestione valori nulli e tipi di dato**: trattamento dei dati mancanti e correzione dei formati.
- **Salvataggio dei CSV puliti**: esportazione dei dati pronti all'uso nella cartella `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
from functions.utils import show_dtypes, check_duplicates, check_missing, clean_duplicates, drop_empty_rows, drop_key_missing, convert_to_datetime

## Caricamento Dataset

In questa sezione carichiamo i file CSV originali forniti per l'analisi.

In [ ]:
characters_df = pd.read_csv("../data/datasets/characters.csv")
character_anime_works_df = pd.read_csv("../data/datasets/character_anime_works.csv")
character_nicknames_df = pd.read_csv("../data/datasets/character_nicknames.csv")
details_df = pd.read_csv("../data/datasets/details.csv")
favs_df = pd.read_csv("../data/datasets/favs.csv")
person_alternate_names_df = pd.read_csv("../data/datasets/person_alternate_names.csv")
person_anime_works_df = pd.read_csv("../data/datasets/person_anime_works.csv")
person_voice_works_df = pd.read_csv("../data/datasets/person_voice_works.csv")
person_details_df = pd.read_csv("../data/datasets/person_details.csv")
profiles_df = pd.read_csv("../data/datasets/profiles.csv")
ratings_df = pd.read_csv("../data/datasets/ratings.csv")
recommendations_df = pd.read_csv("../data/datasets/recommendations.csv")
stats_df = pd.read_csv("../data/datasets/stats.csv")

## Rimozione dei Duplicati

In questa sezione analizziamo l'integrità dei dati cercando eventuali righe duplicate in tutti i dataset tramite la funzione `check_duplicates`.

In [ ]:
# Analizza la presenza di dati duplicati
check_duplicates(characters_df, name="characters")
check_duplicates(character_anime_works_df, name="character_anime_works")
check_duplicates(character_nicknames_df, name="character_nicknames")
check_duplicates(details_df, name="details")
check_duplicates(favs_df, name="favs")
check_duplicates(person_alternate_names_df, name="person_alternate_names")
check_duplicates(person_anime_works_df, name="person_anime_works")
check_duplicates(person_voice_works_df, name="person_voice_works")
check_duplicates(person_details_df, name="person_details")
check_duplicates(profiles_df, name="profiles")
check_duplicates(ratings_df, name="ratings")
check_duplicates(recommendations_df, name="recommendations")
check_duplicates(stats_df, name="stats")

Successivamente, utilizziamo la funzione `clean_duplicates` per rimuoverli definitivamente, garantendo che ogni dataset contenga solo informazioni univoche e che il numero di righe rimosse corrisponda alle copie rilevate prima.

In [ ]:
# Per efficienza usiamo solo i dataset che hanno prodotto duplicati
characters_df = clean_duplicates(characters_df, "Characters")
character_nicknames_df = clean_duplicates(character_nicknames_df, "Character_nicknames")
person_alternate_names_df = clean_duplicates(person_alternate_names_df, "Person_alternate_names")
person_voice_works_df = clean_duplicates(person_voice_works_df, "Person_voice_works")
person_details_df = clean_duplicates(person_details_df, "Person_details")

## Definizione dei Tipi di Dato (dtypes)

In questa sezione impostiamo il formato corretto per le colonne di ogni dataset.   
Iniziamo visualizzando i tipi di dato di ogni dataset utilizzando l'attributo `.dtypes`.

In [ ]:
# La funzione stampa il nome del dataset, per distinguere i risultati
show_dtypes(characters_df, "Characters")
show_dtypes(character_anime_works_df, "Character_anime_works")
show_dtypes(character_nicknames_df, "Character_nicknames")
show_dtypes(details_df, "Details")
show_dtypes(favs_df, "Favs")
show_dtypes(person_alternate_names_df, "Person_alternate_names")
show_dtypes(person_anime_works_df, "Person_anime_works")
show_dtypes(person_voice_works_df, "Person_voice_works")
show_dtypes(person_details_df, "Person_details")
show_dtypes(profiles_df, "Profiles")
show_dtypes(ratings_df, "Ratings")
show_dtypes(recommendations_df, "Recommendations")
show_dtypes(stats_df, "Stats")

Trasformiamo tutte le colonne contenenti date nel formato datetime, utilizzando la funzione `convert_to_datetime` per la gestione dei valori nulli e delle date non valide.

In [ ]:
# Trasformazione in datetime delle colonne nei dataset person_details, profiles e details
person_details_df = convert_to_datetime(person_details_df, "birthday")
profiles_df = convert_to_datetime(profiles_df, "joined")
profiles_df = convert_to_datetime(profiles_df, "birthday")
details_df = convert_to_datetime(details_df, "start_date")
details_df = convert_to_datetime(details_df, "end_date")

Successivamente, convertiamo le restanti colonne numeriche nel tipo più adatto per l'analisi statistica. Nello specifico:
- In `Details`: Le colonne 'years', 'scored_by', 'rank' ed 'episodes' vengono convertite da float a Int64. L'uso di Int64 ci permette di trattare queste colonne come interi pur mantenendo eventuali valori nulli (NaN) senza generare errori.
- In `Profiles`: Le colonne relative allo stato di visione ('watching', 'completed', ecc.) venivano lette come object a causa della virgola usata come separatore delle migliaia. Abbiamo rimosso la virgola per permettere la corretta conversione in Int64.
- In `Characters`: La colonna 'favorites' viene uniformata al tipo Int64.

In [ ]:
# Dataset Details
details_df["year"] = details_df["year"].astype("Int64")
details_df["scored_by"] = details_df["scored_by"].astype("Int64")
details_df["rank"] = details_df["rank"].astype("Int64")
details_df["episodes"] = details_df["episodes"].astype("Int64")

# Dataset Characters
characters_df["favorites"] = characters_df["favorites"].astype("Int64")

# Dataset Profiles
if "watching" in profiles_df.columns:
    profiles_df["watching"] = pd.to_numeric(
        profiles_df["watching"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "completed" in profiles_df.columns:
    profiles_df["completed"] = pd.to_numeric(
        profiles_df["completed"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "on_hold" in profiles_df.columns:
    profiles_df["on_hold"] = pd.to_numeric(
        profiles_df["on_hold"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "dropped" in profiles_df.columns:
    profiles_df["dropped"] = pd.to_numeric(
        profiles_df["dropped"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "plan_to_watch" in profiles_df.columns:
    profiles_df["plan_to_watch"] = pd.to_numeric(
        profiles_df["plan_to_watch"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

## Analisi e Gestione dei Valori Nulli (NaN)

In questa fase finale della pulizia, ci occupiamo dei valori mancanti all'interno dei nostri dataset.  
Utilizziamo la funzione `check_missing` per visualizzare la percentuale dei valori NaN e identificare in quali colonne specifiche si trovano.

In [ ]:
# Calcolo della percentuale di valori mancanti per ogni colonna dei dataset
check_missing(characters_df, "Characters")
check_missing(character_anime_works_df, "Character_anime_works")
check_missing(character_nicknames_df, "Character_nicknames")
check_missing(details_df, "Details")
check_missing(favs_df, "Favs")
check_missing(person_alternate_names_df, "Person_alternate_names")
check_missing(person_anime_works_df, "Person_anime_works")
check_missing(person_voice_works_df, "Person_voice_works")
check_missing(person_details_df, "Person_details")
check_missing(profiles_df, "Profiles")
check_missing(ratings_df, "Ratings")
check_missing(recommendations_df, "Recommendations")
check_missing(stats_df, "Stats")

Attraverso la funzione `drop_empty_rows`, eliminiamo le righe che contengono esclusivamente valori NaN. Queste righe risultano totalmente inutili alla nostra analisi.

In [ ]:
# Rimozione delle righe che contengono solo valori NaN
characters_df = drop_empty_rows(characters_df, "Characters")
character_anime_works_df = drop_empty_rows(character_anime_works_df, "Character_anime_works")
character_nicknames_df = drop_empty_rows(character_nicknames_df, "Character_nicknames")
details_df = drop_empty_rows(details_df, "Details")
favs_df = drop_empty_rows(favs_df, "Favs")
person_alternate_names_df = drop_empty_rows(person_alternate_names_df, "Person_alternate_names")
person_anime_works_df = drop_empty_rows(person_anime_works_df, "Person_anime_works")
person_voice_works_df = drop_empty_rows(person_voice_works_df, "Person_voice_works")
person_details_df = drop_empty_rows(person_details_df, "Person_details")
profiles_df = drop_empty_rows(profiles_df, "Profiles")
ratings_df = drop_empty_rows(ratings_df, "Ratings")
recommendations_df = drop_empty_rows(recommendations_df, "Recommendations")
stats_df = drop_empty_rows(stats_df, "Stats")

Usiamo la funzione `drop_key_missing` per rimuovere le righe senza le colonne "importanti" in ciascun dataset dove `check_missing` ha rilevato valori mancanti.

In [ ]:
# Characters nickname → rimuovi righe senza nickname
character_nicknames_df = drop_key_missing(character_nicknames_df, "nickname", "Character_nicknames")

# Person_alternate_names → rimuovi righe senza alt_name
person_alternate_names_df = drop_key_missing(person_alternate_names_df, "alt_name", "Person_alternate_names")

# Favs → rimuovi righe senza username
favs_df = drop_key_missing(favs_df, "username", "Favs")

# Profiles → rimuovi righe senza username
profiles_df = drop_key_missing(profiles_df, "username", "Profiles")

# Ratings → rimuovi righe senza username
ratings_df = drop_key_missing(ratings_df, "username", "Ratings")


## Salvataggio dei Dataset Puliti

In quest'ultima fase della preparazione, procediamo al salvataggio dei dataset elaborati. Dopo aver rimosso i duplicati, normalizzato i tipi di dato e gestito i valori mancanti, esportiamo i file in una cartella dedicata `data/processed/`.

In [ ]:
characters_df.to_csv("../data/processed/Characters_cleaned.csv", index=False)
character_anime_works_df.to_csv("../data/processed/Character_anime_works_cleaned.csv", index=False)
character_nicknames_df.to_csv("../data/processed/Character_nicknames_cleaned.csv", index=False)
details_df.to_csv("../data/processed/Details_cleaned.csv", index=False)
favs_df.to_csv("../data/processed/Favs_cleaned.csv", index=False)
person_alternate_names_df.to_csv("../data/processed/Person_alternate_names_cleaned.csv", index=False)
person_anime_works_df.to_csv("../data/processed/Person_anime_works_cleaned.csv", index=False)
person_voice_works_df.to_csv("../data/processed/Person_voice_works_cleaned.csv", index=False)
person_details_df.to_csv("../data/processed/Person_details_cleaned.csv", index=False)
profiles_df.to_csv("../data/processed/Profiles_cleaned.csv", index=False)
ratings_df.to_csv("../data/processed/Ratings_cleaned.csv", index=False)
recommendations_df.to_csv("../data/processed/Recommendations_cleaned.csv", index=False)
stats_df.to_csv("../data/processed/Stats_cleaned.csv", index=False)